In [ ]:
import numpy as np
from  numpy import deg2rad as d2r
from  numpy import array as arr

from scipy.optimize import minimize, approx_fprime
from cyipopt import minimize_ipopt as cyminimize
from space_traj_opt.models3d import dynamics
from space_traj_opt.controller3d import  CtrlMode, flight_path_angle, lts_control

from space_traj_opt.transcription import MultiShootingTranscription
from space_traj_opt.utils import unpack_sol_list
from space_traj_opt.plotting import plot, visualize_jac2

from space_traj_opt.math.integrator import integrate

STANDARD_GRAV = 9.80665

## Electron Rocket Parameters

In [ ]:
n_engines_s1 = 9
n_engines_s2 = 1
isp_s1 = 311.0
engine_thrust_s1 = n_engines_s1*24910.04  # N Average between sl and vac
isp_s2 = 343.0
engine_thrust_s2 = n_engines_s2* 25_000.0  # N
s1_vch_params = (engine_thrust_s1, isp_s1)
s2_vch_params = (engine_thrust_s2, isp_s2)

fairing_mass = 50.0
farinig_timing = 184.0 - 162.0 # sec
payload = 250.0
s1_dry_mass = 1076.47308279  
s2_dry_mass = 257.90093739  

s1_wet_mass = 10047.082106064723
s2_wet_mass = 2602.454913676189
total_mass = 12949.537019740912

mdot_s1 = engine_thrust_s1 / STANDARD_GRAV / isp_s1
mdot_s2 = engine_thrust_s2 / STANDARD_GRAV / isp_s2

In [ ]:
NUM_X= 7
NUM_U = 4
NUM_PHASE = 1
# %load_ext snakeviz

## Initial Guesses

In [ ]:
mu_earth = 3.986004418e14
earth_r = 6_378_000.0 # m
circ_orbit_alt = 200_000.0 
v_circ = np.sqrt(mu_earth / (earth_r + circ_orbit_alt))

# Guesses 
s2_sep_mass = s2_wet_mass + payload + fairing_mass
x0 = arr([
    [0, 84918 + earth_r,0, 2457, 997, 0, s2_wet_mass + payload - farinig_timing * mdot_s2]
])


## normalization vector 
x0_n_vec = arr([earth_r, earth_r, earth_r, 5000, 1000, 1000, 5000])

In [ ]:
x0 


## Define a multiphase trajectory problem

In [ ]:
problem = MultiShootingTranscription(["phase0"], NUM_X, dynamics)

problem.set_dynamics_params("phase0", s2_vch_params)


## Define state, control and time guesses for each phase 

In [ ]:

problem.set_phase_init_x("phase0", x0 = x0[0], norm_vec = x0_n_vec, bounds = x0[0])
problem.set_phase_control(
    "phase0", 
    CtrlMode.LTS, 
    u0 = arr([-0.0017,0.5, 0,0]), bounds = [(-0.1,0.1), (-3,3), (-0.1,0.1), (-3,3)], norm_vec =[0.1,np.pi/2, 0.1,np.pi/2])
problem.set_phase_time("phase0", t0 = 298)

a_desired = circ_orbit_alt + earth_r
e_desired = 0.0

x_f = arr([a_desired, e_desired, s2_dry_mass + payload])
xf_n_vec = arr([a_desired, 1.0, 5000])
problem.set_terminal_state(x_final = x_f, bounds = arr([a_desired, None, None]), norm_vec = xf_n_vec)

## Build The problem
Builds the decision vector and bounds 

In [ ]:
d0, d_bounds, normalization_vec, full_params = problem.build()
d0_norm, d_bounds_norm = problem.normalize_decision_vec(d0, d_bounds,normalization_vec)

In [ ]:
config = full_params[0]
u, x, t_terminal, control_law = problem.unpack_decision_var(d0, config=config)


In [ ]:

# make inputs hashable, needed for lru cache, the copy is cheaper than a second f(x) eval
u_ = tuple(u.tolist())
x_ = tuple(x.tolist())
t_ = float(t_terminal)
vch_params = (config[3], (control_law, u_))

problem.traj_rollout(t_, x_, vch_params)

In [ ]:


t_sol, state = integrate(
    dynamics, 
    t_span=[0.0, t_], 
    t_eval= np.linspace(0.0, t_,150),
    y0=x_,    
    args=(vch_params,)
)

In [ ]:
#state = sol.y

pos = state[0:3]
vel = state[3:6]
mass = state[6]

In [ ]:
np.linalg.norm(pos.T[0]) - earth_r

In [ ]:
alt_geocentric = np.linalg.norm(pos, axis=0) - earth_r

v_eci_mag = np.linalg.norm(vel, axis=0) 

In [ ]:

ctr_param, x0, time, mode = problem.unpack_decision_var(d0, full_params[0] )
ctr_param

In [ ]:
from space_traj_opt.math.orbital_calcs import rv_to_aei

a_values = []
e_values = []
i_values = []
fpa_values = []
lts_values = []
pitch_values = []
for p, v in zip(pos.T, vel.T):
    a, e, i = rv_to_aei(p, v, mu_earth)
    a_values.append(a)
    e_values.append(e)
    i_values.append(i)
    fpa_values.append(flight_path_angle(p, v))

for t, cur_x in zip(t_sol, state.T):
    lts_params = lts_control(t, cur_x,ctr_param )
    lts_values.append(lts_params)
    pitch_values.append(np.atan2(lts_params[1], lts_params[0]) )

a = np.array(a_values)
e = np.array(e_values)
i = np.array(i_values)
fpa = np.array(fpa_values)
pitch = np.array(pitch_values)

r_periapsis = a * (1.0 - e)
r_apoapsis  = a * (1.0 + e)

h_periapsis = r_periapsis - earth_r
h_apoapsis  = r_apoapsis  - earth_r



In [ ]:
plot(
    t_sol, [fpa, pitch],
    title="Time vs States", 
    xlabel="Time", 
    ylabel="Flight Path Angle, Pitch_RSW",
    )

In [ ]:
plot(
    t_sol, [h_periapsis, h_apoapsis],
    title="Time vs States", 
    xlabel="Time", 
    ylabel="perigee and apogee altitudes",
    )

In [ ]:
plot(
    t_sol, e,
    title="Time vs States", 
    xlabel="Time", 
    ylabel="Eccentricity",
    )

In [ ]:
plot(
    t_sol, mass,
    title="Time vs States", 
    xlabel="Time", 
    ylabel="Mass",
    )

In [ ]:
plot(
    t_sol, v_eci_mag,
    title="Time vs States", 
    xlabel="Time", 
    ylabel="vel mag",
    )